In [292]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp

In [293]:
# Wind speed and angle
WIND_SPEED_MULTIPLIER = 10
CURRENT_MULTIPLIER = 0.1
PHI = -3*np.pi/4  # The angle of the wind

In [294]:
# ─────────────────────────────────────────────
# Sailing polar: boat speed as a function of
# the angle between heading and the wind.
#
# wind_dir: angle (rad) the wind blows TOWARD
# alpha   : angle between boat heading and wind_dir
#
# No-go zone: |alpha| < no_go  → speed = 0
# Peak speed at alpha ≈ 60–90° off the wind
# Tapers off going dead downwind
# ─────────────────────────────────────────────

no_go = np.radians(45)   # half-width of the upwind no-go zone

# Wind blows FROM the top-right (northeast) TOWARD the bottom-left (southwest)
# phi is defined in your code as the direction the wind blows TOWARD,
# so the upwind direction (where wind comes from) is phi + π

def boat_speed(heading, wind_dir):
    """
    Return achievable boat speed for a given heading.

    heading  : angle the boat moves TOWARD (rad)
    wind_dir : angle the wind blows TOWARD (rad)  ← your phi
    """
    # Angle of heading relative to the upwind direction (where wind comes FROM)
    # alpha = 0  → pointing dead into the wind (forbidden)
    # alpha = π  → running dead downwind
    upwind_dir = wind_dir + np.pi          # direction wind comes FROM
    alpha = heading - upwind_dir
    alpha = (alpha + np.pi) % (2 * np.pi) - np.pi   # wrap to [-π, π]

    # No-go zone: can't sail within 45° of the upwind direction
    no_go = np.radians(45)
    if np.abs(alpha) < no_go:
        return 0.0

    # Polar: zero at α=0 (into wind), peaks ~70° off wind, tapers downwind
    speed = WIND_SPEED_MULTIPLIER * (np.sin(alpha) ** 2) * (1 + np.cos(alpha) ** 2) / 2
    return float(np.clip(speed, 0, WIND_SPEED_MULTIPLIER))


# ─────────────────────────────────────────────
# Drift (ocean current), unchanged
# ─────────────────────────────────────────────
def w(x, y):
    x, y = np.atleast_1d(x), np.atleast_1d(y)
    return np.vstack((5 * y, np.zeros_like(y)))

def dw(x, y):
    return np.array([
        [0, 2],
        [0, 0]
    ])


# ─────────────────────────────────────────────
# ODE — optimal control with sailing polar
#
# The costate vector λ = (l1, l2) still points
# in the direction the ship WANTS to go.
# We compute the best achievable heading by
# maximising λ · u subject to the polar constraint.
#
# Because the polar is not symmetric and has a
# no-go zone, we sample candidate headings and
# pick the one that maximises the Hamiltonian.
# ─────────────────────────────────────────────
def best_heading(l1, l2, wind_dir):
    """
    Vectorized: l1, l2 can be scalars or 1-D arrays.
    Returns (u1, u2) arrays of the same length.
    """
    l1 = np.atleast_1d(l1)
    l2 = np.atleast_1d(l2)
    n  = l1.shape[0]

    # Sample headings, pre-compute polar speeds for all of them
    thetas = np.linspace(-np.pi, np.pi, 720, endpoint=False)
    speeds = np.array([boat_speed(th, wind_dir) for th in thetas])  # shape (720,)

    # u vectors for each candidate heading: shape (720,)
    u1_candidates = speeds * np.cos(thetas)
    u2_candidates = speeds * np.sin(thetas)

    # For each point i, find the heading that maximises λ·u
    # λ·u = l1[i]*u1 + l2[i]*u2  → shape (n, 720) via broadcasting
    dot = l1[:, None] * u1_candidates[None, :] + l2[:, None] * u2_candidates[None, :]  # (n, 720)

    best_idx = np.argmax(dot, axis=1)  # (n,)

    u1_out = u1_candidates[best_idx]
    u2_out = u2_candidates[best_idx]

    return u1_out, u2_out


def ode(t, Y, p):
    T        = p[0]
    x, y, l1, l2 = Y          # all shape (n,)

    u1, u2   = best_heading(l1, l2, PHI)

    w1, w2   = w(x, y)

    dx = T * (u1 + w1)
    dy = T * (u2 + w2)

    # Dw is constant here, so the costate equation is still vectorised cleanly
    Dw       = dw(x, y)                          # (2,2)
    lam      = np.vstack([l1, l2])               # (2, n)
    dlam     = -T * (Dw.T @ lam)                 # (2, n)
    dl1, dl2 = dlam[0], dlam[1]

    return np.array([dx, dy, dl1, dl2])


# ─────────────────────────────────────────────
# Boundary conditions — updated to use polar
# ─────────────────────────────────────────────
def bc(ya, yb, p):
    T = p[0]

    xa,  ya_val, l1a, l2a = ya
    xb,  yb_val, l1b, l2b = yb

    u1b, u2b = best_heading(l1b, l2b, PHI)
    wb       = w(xb, yb_val).flatten()

    # Transversality: H = 1 at the free final time
    Hf = l1b * (u1b + wb[0]) + l2b * (u2b + wb[1]) - 1.0

    return np.array([
        xa    - x0[0],
        ya_val - x0[1],
        xb    - xf[0],
        yb_val - xf[1],
        Hf
    ])


# ─────────────────────────────────────────────
# Solve (unchanged structure)
# ─────────────────────────────────────────────
t = np.linspace(0, 1, 100)

Y_guess      = np.zeros((4, t.size))
p_guess      = np.array([2.0])
Y_guess[0]   = np.linspace(x0[0], xf[0], t.size)
Y_guess[1]   = np.linspace(x0[1], xf[1], t.size)
Y_guess[2]   = 1.0
Y_guess[3]   = 1.0

sol = solve_bvp(ode, bc, t, Y_guess, p=p_guess)

# ─────────────────────────────────────────────
# Plot
# ─────────────────────────────────────────────
plt.plot(sol.y[0], sol.y[1], label="Optimal Path")
plt.arrow(0, 0, np.cos(PHI), np.sin(PHI),
          head_width=0.1, head_length=0.1,
          fc="red", ec="red", label=f"Wind dir (speed {v})")

x_grid = np.linspace(0, 4, 10)
X, Y_g = np.meshgrid(x_grid, x_grid)
U_q    = 5 * Y_g
V_q    = np.zeros_like(X)
plt.quiver(X, Y_g, U_q, V_q)

plt.title("Optimal Sailing Path with Realistic Wind Polar")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (5,) + inhomogeneous part.